# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jessicaayegh/Ml-notebook/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.CONTRACT (5 answers)

1. What one row means: In the warehouse table, one row is one report date for one client and one content item. In my modeling frame, I collapse March into one row per content item (client + content).

2. Tables: fact_content_daily_performance (the month=2026-03 file) for search impressions, clicks and position, and dim_content for the content's created date.

3. Time window: Features use 1-15 March 2026. The label uses 16-30 March 2026. The decision moment is the end of 15 March. June 2026 stays sealed and untouched.

4. What I rank/predict: Which content is declining. Proxy label: declining = 1 if the content got fewer search impressions in 16-30 March than in 1-15 March, using only content with at least 10 impressions in 1-15 March. I rank content by predicted probability of declining.

5. Deliberate exclusion: The dim_content date fields content_updated_date, last_optimized_date and optimization_eligible_date. They are snapshot values from the July 2026 export (the sample rows show updates dated July 2026), so they can contain information from after my March decision moment.

In [31]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verify grain: no duplicate (client, content_id, date) rows in March 2026



## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*
FIELDS

Features (knowable by 15 March): impressions_h1, clicks_h1, avg_position_h1, active_days_h1, content_age_days
Label / proxy: impressions_h2 (16-30 March impressions) and declining, which is computed from it
Context (never fed to the model): client_hash_id, content_hash_id, report_date
Excluded: content_updated_date, last_optimized_date, optimization_eligible_date (export-time values, may postdate March)
Not used: GA4 columns (blank in the sample rows I inspected)

In [32]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Query 1 of 3 — GRAIN: is one row really one (report_date, client, content item)?
q1 = con.execute("""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
FROM perf_mar
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
""").fetchdf()
print(f"Duplicate (date, client, content) combinations found: {len(q1)}")
q1

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate (date, client, content) combinations found: 0


,report_date,client_hash_id,content_hash_id,n


In [25]:
# Query 2 of 3 — SLICE SIZE: row count and date span for March 2026
q2 = con.execute("""
SELECT COUNT(*)                        AS row_count,
       MIN(report_date)                AS first_date,
       MAX(report_date)                AS last_date,
       COUNT(DISTINCT report_date)     AS distinct_days,
       COUNT(DISTINCT client_hash_id)  AS clients,
       COUNT(DISTINCT content_hash_id) AS content_items
FROM perf_mar
""").fetchdf()
q2

,row_count,first_date,last_date,distinct_days,clients,content_items
0,9841378,2026-03-01,2026-03-31,31,55,331437


In [26]:
# Query 3 of 3 — AVAILABILITY: how many rows survive gsc_data_available IS TRUE?
q3 = con.execute("""
SELECT COUNT(*) AS total_rows,
       COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS rows_surviving_is_true,
       ROUND(100.0 * COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) / COUNT(*), 2) AS pct_surviving
FROM perf_mar
""").fetchdf()
q3


,total_rows,rows_surviving_is_true,pct_surviving
0,9841378,3611061,36.69


In [27]:
# Build the feature frame: one row per content item, features from 1-15 March, label from 16-30 March.
frame = con.execute("""
WITH perf AS (
  SELECT * FROM perf_mar WHERE gsc_data_available IS TRUE
),
h1 AS (
  SELECT client_hash_id, content_hash_id,
         CAST(SUM(gsc_impressions) AS BIGINT) AS impressions_h1,
         CAST(SUM(gsc_clicks) AS BIGINT)      AS clicks_h1,
         SUM(gsc_sum_position) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS avg_position_h1,
         COUNT(DISTINCT report_date) FILTER (WHERE gsc_impressions > 0) AS active_days_h1
  FROM perf
  WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
  GROUP BY client_hash_id, content_hash_id
),
h2 AS (
  SELECT client_hash_id, content_hash_id,
         CAST(SUM(gsc_impressions) AS BIGINT) AS impressions_h2
  FROM perf
  WHERE report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-30'
  GROUP BY client_hash_id, content_hash_id
),
base AS (
  SELECT h1.client_hash_id, h1.content_hash_id,
         h1.impressions_h1, h1.clicks_h1, h1.avg_position_h1, h1.active_days_h1,
         COALESCE(h2.impressions_h2, 0) AS impressions_h2,
         DATE_DIFF('day', c.content_created_date, DATE '2026-03-15') AS content_age_days
  FROM h1
  LEFT JOIN h2 ON h1.client_hash_id = h2.client_hash_id
              AND h1.content_hash_id = h2.content_hash_id
  JOIN dim_content c ON h1.client_hash_id = c.client_hash_id
                    AND h1.content_hash_id = c.content_hash_id
  WHERE h1.impressions_h1 >= 10
)
SELECT * FROM base WHERE content_age_days >= 0
""").fetchdf()

frame["declining"] = (frame["impressions_h2"] < frame["impressions_h1"]).astype(int)

print("Rows in feature frame:", len(frame))
print("Share labelled declining:", round(frame["declining"].mean(), 3))
frame.drop(columns=["client_hash_id", "content_hash_id"]).describe().T


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows in feature frame: 120513
Share labelled declining: 0.478


,count,mean,std,min,25%,50%,75%,max
impressions_h1,120513.0,1057.149569,2966.915468,10.0,55.00000,216.000000,856.000000,161575.000000
clicks_h1,120513.0,3.185864,14.798697,0.0,0.00000,0.000000,2.000000,2395.000000
avg_position_h1,120513.0,15.528818,16.806753,0.0,4.78553,8.336683,20.165605,119.429688
active_days_h1,120513.0,12.964079,3.181839,1.0,12.00000,15.000000,15.000000,15.000000
impressions_h2,120513.0,1147.385452,3500.385160,0.0,60.00000,234.000000,936.000000,420943.000000
content_age_days,120513.0,180.103391,124.562735,1.0,58.00000,177.000000,250.000000,478.000000
declining,120513.0,0.477567,0.499499,0.0,0.00000,0.000000,1.000000,1.000000


FIVE FEATURES — "knowable at the decision moment because…"

impressions_h1: total search impressions 1-15 March, all observed before the 15 March decision moment.
clicks_h1: total clicks over the same 1-15 March window, so it is already recorded by the decision moment.
avg_position_h1: impression-weighted average search position over 1-15 March, computed only from rows dated on or before 15 March.
active_days_h1: number of days in 1-15 March with at least one impression, again observed before the decision moment.
content_age_days: 15 March minus the content's created date, and a creation date is fixed once the content exists. I keep only content created on or before 15 March (age of 0 or more).

In [28]:
# The honest model: a quick score using only the five safe features.
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

FEATURES = ["impressions_h1", "clicks_h1", "avg_position_h1", "active_days_h1", "content_age_days"]

def quick_score(feature_list):
    # Split by client so the same client never appears in both train and test.
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
    train_idx, test_idx = next(splitter.split(frame, groups=frame["client_hash_id"]))
    train, test = frame.iloc[train_idx], frame.iloc[test_idx]
    model = HistGradientBoostingClassifier(random_state=42)
    model.fit(train[feature_list], train["declining"])
    return roc_auc_score(test["declining"], model.predict_proba(test[feature_list])[:, 1])

honest_auc = quick_score(FEATURES)
print(f"HONEST score (AUC, 5 safe features): {honest_auc:.3f}")

HONEST score (AUC, 5 safe features): 0.570


In [29]:
# The deliberate leak: add ONE label-derived column (impressions_h2, the 16-30 March impressions).
leaky_auc = quick_score(FEATURES + ["impressions_h2"])
print(f"LEAKY score (AUC, with impressions_h2): {leaky_auc:.3f}")

# Remove it. The honest number is the one we keep.
print(f"After removing the leak, the honest score is still: {quick_score(FEATURES):.3f}")

LEAKY score (AUC, with impressions_h2): 1.000
After removing the leak, the honest score is still: 0.570


LEAKAGE EXPERIMENT

LEAKAGE EXPERIMENT

I added impressions_h2 (impressions from 16-30 March) as a feature on purpose. The label is "impressions_h2 < impressions_h1", so this column gives the model the answer. The score (AUC) jumped from 0.573 to 1.000, a perfect score. I deleted the column and kept the honest score of 0.573.

What the honest number means: the five features I could really know on 15 March predict the label only a little better than a coin flip (AUC 0.573 on 120,513 content items, 47.8% of them declining). I report that as the result, not as something to fix by adding features I could not have known at the time.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*
LIMITATION

LIMITATION

Only 36.69% of March rows have Search Console data available (gsc_data_available IS TRUE), and only 55 of the warehouse's 104 clients appear in March. My features and label are built only from available rows, so the results describe content with usable Search Console data, not all content. The "declining" label also treats a content item with no available rows in 16-30 March as zero impressions, which may sometimes mean "data not recorded" rather than "no impressions".

In [30]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.